# R1 Development-Batch Segment Inference

- This notebook uses the four-model direct baseline: one scope classifier and three skill classifiers.
- It runs locally over every R1 document in the configured held-out test cohort, using text-only modeling-view columns; it does not send reflection text to a model provider.
- Each document receives its own predicted-analysis artifact for G2. Candidate annotation rows are deduplicated only after confirming identical segment text and order; no gold labels are loaded.

In [12]:
import gc
import json
import os
import tempfile
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import torch
from dotenv import load_dotenv
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

from reflection_assessment_feedback import (
    EXPERT_RUBRIC,
    IntermediateAnalysis,
    ReflectionDocument,
    ReflectionSegment,
    SegmentAnalysis,
    make_generation_request,
)
from reflection_assessment_feedback.experiment_config import load_experiment_config

PROJECT_ROOT = Path.cwd().resolve()
assert PROJECT_ROOT.name == "prebi_v1", "Run from the prebi_v1 project root."
load_dotenv(PROJECT_ROOT / ".env", override=False)
EXPERIMENT_CONFIG = load_experiment_config()
CLASSIFIER_CONFIG = EXPERIMENT_CONFIG["classifier"]
DATASET_CONFIG = EXPERIMENT_CONFIG["dataset"]
SPLIT_PATH = (
    PROJECT_ROOT
    / EXPERIMENT_CONFIG["paths"]["provisional_split_directory"]
    / f"provisional_{DATASET_CONFIG['prediction_split']}_modeling_wide.csv"
)
CHECKPOINT_DIR = PROJECT_ROOT / EXPERIMENT_CONFIG["paths"]["classifier_checkpoints"]
MODEL_NAME = CLASSIFIER_CONFIG["model_name"]
MODEL_REVISION = CLASSIFIER_CONFIG["model_revision"] or None
MAX_LENGTH = CLASSIFIER_CONFIG["max_length"]
BATCH_SIZE = CLASSIFIER_CONFIG["batch_size"]
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available() else "cpu"
)
SKILL_DIMENSIONS = EXPERIMENT_CONFIG["labels"]["skill_dimensions"]
MODEL_LABELS = {"scope": 2, **{f"direct-{skill}": 4 for skill in SKILL_DIMENSIONS}}
print(f"Inference device: {DEVICE}")

Inference device: mps


## Resolve protected paths and the R1 development cohort

The R1 essay selector and held-out split are read from `config/experiment.toml`. `PREBI_DATA_ROOT` must be an approved absolute directory outside this source tree. Each development document is written to its own file under `g2/predictions/`.

In [13]:
protected_root_value = os.getenv("PREBI_DATA_ROOT")
prediction_dir = None
if protected_root_value:
    protected_root = Path(protected_root_value).expanduser()
    if not protected_root.is_absolute():
        raise ValueError("PREBI_DATA_ROOT must be an absolute path outside the project.")
    protected_root = protected_root.resolve()
    if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
        raise ValueError("PREBI_DATA_ROOT must be outside the project source tree.")
    prediction_dir = protected_root / "g2" / "predictions"
    prediction_dir.mkdir(parents=True, exist_ok=True, mode=0o700)
else:
    print("Set PREBI_DATA_ROOT to an approved protected directory before writing.")

feedback_path = (PROJECT_ROOT / EXPERIMENT_CONFIG["paths"]["human_feedback"]).resolve()
feedback_rows = pd.read_csv(
    feedback_path,
    usecols=["essay_name", "document_id"],
    dtype={"document_id": str},
)
feedback_rows["essay_name"] = feedback_rows["essay_name"].astype(str).str.strip().str.upper()
feedback_rows["document_id"] = feedback_rows["document_id"].astype(str).str.strip()
split_document_ids = set(
    pd.read_csv(SPLIT_PATH, usecols=["document_id"], dtype={"document_id": str})[
        "document_id"
    ].dropna()
)
DEVELOPMENT_DOCUMENT_IDS = sorted(
    set(
        feedback_rows.loc[
            feedback_rows["essay_name"] == DATASET_CONFIG["development_essay_name"],
            "document_id",
        ]
    )
    & split_document_ids
)
if not DEVELOPMENT_DOCUMENT_IDS:
    raise ValueError("No development-cohort documents are available in the selected split.")
if prediction_dir is not None:
    OUTPUT_PATHS = {
        document_id: prediction_dir / f"predicted-analysis-{quote(document_id, safe='')}.json"
        for document_id in DEVELOPMENT_DOCUMENT_IDS
    }
else:
    OUTPUT_PATHS = {}
print(f"Selected {len(DEVELOPMENT_DOCUMENT_IDS)} development documents for inference.")

Selected 6 development documents for inference.


## Load R1 document segments for inference

Read only IDs, text, and source order for each configured R1 document. Candidate rows may repeat a segment; require their text and order to agree, then create one model input per unique segment. Human annotation targets are not loaded.

In [14]:
reflection_rows = pd.read_csv(
    SPLIT_PATH,
    usecols=["document_id", "segment_id", "text", "segment_order"],
    dtype={"document_id": str, "segment_id": str},
)
reflection_rows["text"] = reflection_rows["text"].fillna("").astype(str)
reflection_rows["segment_order"] = pd.to_numeric(
    reflection_rows["segment_order"], errors="raise"
).astype(int)
reflection_rows = reflection_rows.loc[
    reflection_rows["document_id"].isin(DEVELOPMENT_DOCUMENT_IDS)
].copy()
if reflection_rows[["segment_id", "text", "segment_order"]].isna().any().any():
    raise ValueError("Selected development documents contain missing segment IDs, text, or order.")

DEVELOPMENT_DOCUMENTS = {}
for selected_document_id, candidate_rows in reflection_rows.groupby("document_id", sort=True):
    consistency = candidate_rows.groupby("segment_id").agg(
        text_variants=("text", "nunique"),
        order_variants=("segment_order", "nunique"),
    )
    if (consistency > 1).any().any():
        raise ValueError("Candidate copies disagree on segment text or source order.")
    document_rows = (
        candidate_rows.drop_duplicates("segment_id")
        .sort_values("segment_order")
        .reset_index(drop=True)
    )
    DEVELOPMENT_DOCUMENTS[selected_document_id] = ReflectionDocument(
        document_id=selected_document_id,
        segments=[
            ReflectionSegment(
                segment_id=row.segment_id,
                text=row.text,
                order=row.segment_order,
            )
            for row in document_rows.itertuples(index=False)
        ],
    )
if set(DEVELOPMENT_DOCUMENTS) != set(DEVELOPMENT_DOCUMENT_IDS):
    raise ValueError("Some configured development documents have no segment rows.")
print(
    f"Loaded {len(DEVELOPMENT_DOCUMENTS)} development documents and "
    f"{sum(len(document.segments) for document in DEVELOPMENT_DOCUMENTS.values())} ordered segments; text withheld."
)

Loaded 6 development documents and 282 ordered segments; text withheld.


## Restore checkpointed model artifacts

The four completed direct-baseline runs must exist in `artifacts/segment-classification/`: `scope` and one `direct-<skill>` run for each skill. A bare `.pt` file is not a completed run. The tokenizer and base configuration match the training notebook (`deepset/gbert-base`, revision `None`, length 256); strict weight loading checks the architecture and label head. Change these constants only together with the training configuration.

In [15]:
RUN_INFERENCE = True


def completed_run(run_name, expected_labels):
    metadata_path = CHECKPOINT_DIR / f"{run_name}.json"
    checkpoint_path = CHECKPOINT_DIR / f"{run_name}.pt"
    if not metadata_path.is_file() or not checkpoint_path.is_file():
        raise FileNotFoundError(f"Complete training first: {run_name} requires both .json and .pt artifacts.")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if (
        metadata.get("num_labels") != expected_labels
        or Path(metadata.get("checkpoint", "")).name != checkpoint_path.name
        or not metadata.get("history")
    ):
        raise ValueError(f"Incomplete or incompatible checkpoint metadata for {run_name}.")
    return checkpoint_path


def restore_model(run_name, expected_labels):
    config = AutoConfig.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, num_labels=expected_labels)
    if config.model_type != "bert" or config.num_labels != expected_labels:
        raise ValueError(f"Unexpected model configuration for {run_name}.")
    model = AutoModelForSequenceClassification.from_config(config)
    state = torch.load(checkpoints[run_name], map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE).eval()


checkpoints = {}
tokenizer = None
if RUN_INFERENCE:
    checkpoints = {
        run_name: completed_run(run_name, count)
        for run_name, count in MODEL_LABELS.items()
    }
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    print("Four completed checkpoints and tokenizer are ready.")
else:
    print("Inference disabled; set RUN_INFERENCE = True after training all four direct-baseline models.")

Four completed checkpoints and tokenizer are ready.


## Run batched segment-level prediction

Predict scope for every unique segment, then run the direct skill models only for predicted in-scope segments. Keep logits/probabilities in memory only; the G2 contract stores categorical labels, not raw text or confidence scores.

In [16]:
def predict_logits(run_name, texts):
    if tokenizer is None:
        raise RuntimeError("Enable RUN_INFERENCE and load the tokenizer first.")
    if not texts:
        return torch.empty((0, MODEL_LABELS[run_name]))
    model = restore_model(run_name, MODEL_LABELS[run_name])
    batches = []
    with torch.inference_mode():
        for start in range(0, len(texts), BATCH_SIZE):
            encoded = tokenizer(
                texts[start:start + BATCH_SIZE],
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors="pt",
            )
            logits = model(**{key: value.to(DEVICE) for key, value in encoded.items()}).logits
            batches.append(logits.cpu())
    del model
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    return torch.cat(batches)


FLAT_SEGMENT_REFS = [
    (document_id, segment)
    for document_id, development_document in DEVELOPMENT_DOCUMENTS.items()
    for segment in development_document.segments
]
SEGMENT_TEXTS = [segment.text for _, segment in FLAT_SEGMENT_REFS]
SCOPE_PREDICTIONS = []
SKILL_PREDICTIONS = {}
if RUN_INFERENCE:
    scope_logits = predict_logits("scope", SEGMENT_TEXTS)
    SCOPE_PREDICTIONS = scope_logits.argmax(dim=-1).tolist()
    in_scope_indices = [
        index for index, label in enumerate(SCOPE_PREDICTIONS) if label == 1
    ]
    for skill in SKILL_DIMENSIONS:
        if in_scope_indices:
            skill_logits = predict_logits(
                f"direct-{skill}",
                [SEGMENT_TEXTS[index] for index in in_scope_indices],
            )
            SKILL_PREDICTIONS[skill] = dict(
                zip(in_scope_indices, skill_logits.argmax(dim=-1).tolist(), strict=True)
            )
        else:
            SKILL_PREDICTIONS[skill] = {}
    print(
        f"Predicted scope for {len(SCOPE_PREDICTIONS)} segments across "
        f"{len(DEVELOPMENT_DOCUMENTS)} documents; text withheld."
    )
else:
    print("Prediction skipped.")

Predicted scope for 282 segments across 6 documents; text withheld.


## Build `IntermediateAnalysis` from predictions

Scope `0` maps to `out_of_scope` with `N` on all three dimensions. Scope `1` maps to `in_scope` and direct classes `0` through `3` become the corresponding string band on `SW`, `UA`, and `HA`. Validate full ordered coverage against the same G2 request contract used by the generation notebook.

In [17]:
def build_predicted_analysis(document, scope_labels, bands_by_skill):
    if len(scope_labels) != len(document.segments):
        raise ValueError("Scope predictions must cover every segment exactly once.")
    segments = []
    for index, (segment, scope_label) in enumerate(zip(document.segments, scope_labels, strict=True)):
        if scope_label == 0:
            scope = "out_of_scope"
            component_bands = {dimension: "N" for dimension in SKILL_DIMENSIONS.values()}
        elif scope_label == 1:
            scope = "in_scope"
            component_bands = {
                dimension: str(bands_by_skill[skill][index])
                for skill, dimension in SKILL_DIMENSIONS.items()
            }
        else:
            raise ValueError("Scope predictions must be class 0 or 1.")
        segments.append(SegmentAnalysis.model_validate({
            "segment_id": segment.segment_id,
            "scope": scope,
            "component_bands": component_bands,
        }))
    analysis = IntermediateAnalysis.model_validate({"source": "predicted", "segments": segments})
    make_generation_request(condition="G2", document=document, rubric=EXPERT_RUBRIC, analysis=analysis)
    return analysis


PREDICTED_ANALYSES = {}
if RUN_INFERENCE:
    offset = 0
    for document_id, development_document in DEVELOPMENT_DOCUMENTS.items():
        segment_count = len(development_document.segments)
        local_scope = SCOPE_PREDICTIONS[offset:offset + segment_count]
        local_skill_predictions = {}
        for skill in SKILL_DIMENSIONS:
            local_skill_predictions[skill] = {
                local_index: SKILL_PREDICTIONS[skill][offset + local_index]
                for local_index, scope_label in enumerate(local_scope)
                if scope_label == 1
            }
        PREDICTED_ANALYSES[document_id] = build_predicted_analysis(
            development_document,
            local_scope,
            local_skill_predictions,
        )
        offset += segment_count
    if offset != len(SCOPE_PREDICTIONS):
        raise ValueError("Predictions were not partitioned exactly across the development documents.")
    print(f"Validated predicted analyses for {len(PREDICTED_ANALYSES)} development documents.")
else:
    print("Analysis assembly skipped.")

Validated predicted analyses for 6 development documents.


## Write document-scoped JSON to `PREBI_DATA_ROOT/g2/predictions/`

Set `WRITE_PREDICTIONS = True` only after reviewing the selected document and approved protected storage. Each URL-escaped document ID has a separate `predicted-analysis-<document-id>.json` artifact. The writer stages data in a private temporary file and atomically replaces only that document's predicted artifact; it refuses to replace a different document's or a non-predicted artifact. G2 resolves the matching file from the selected document ID.

In [18]:
import hashlib
import json
import os
import tempfile
from pathlib import Path
from urllib.parse import quote

from reflection_assessment_feedback.experiment_config import experiment_config_sha256


def write_predicted_package(output_path, package):
    if output_path.is_file():
        existing = json.loads(output_path.read_text(encoding="utf-8"))
        if existing.get("document_id") != package["document_id"] or existing.get("analysis", {}).get("source") != "predicted":
            raise ValueError("Refusing to replace another document's or a non-predicted artifact.")
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=output_path.parent,
            prefix=".predicted-analysis-", suffix=".tmp", delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
            json.dump(package, temporary_file, ensure_ascii=False, indent=2)
            temporary_file.write("\n")
        os.chmod(temporary_path, 0o600)
        os.replace(temporary_path, output_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


WRITE_PREDICTIONS = True
PREDICTION_PACKAGES = {}
if WRITE_PREDICTIONS:
    if set(PREDICTED_ANALYSES) != set(DEVELOPMENT_DOCUMENT_IDS):
        raise RuntimeError("Predictions do not cover the selected development documents.")
    if not OUTPUT_PATHS:
        raise RuntimeError("Configure PREBI_DATA_ROOT before writing predictions.")
    for document_id, predicted_analysis in PREDICTED_ANALYSES.items():
        package = {
            "document_id": document_id,
            "analysis": predicted_analysis.model_dump(mode="json"),
            "experiment_config_sha256": experiment_config_sha256(),
        }
        if package["document_id"] != document_id:
            raise ValueError("Prediction package does not match its selected document.")
        IntermediateAnalysis.model_validate(package["analysis"])
        write_predicted_package(OUTPUT_PATHS[document_id], package)
        PREDICTION_PACKAGES[document_id] = package
    print(f"Wrote predicted analysis for {len(PREDICTION_PACKAGES)} development documents.")
else:
    print("JSON write skipped. Set WRITE_PREDICTIONS = True when ready.")

Wrote predicted analysis for 6 development documents.


In [ ]:
prediction_package

## Sanity checks and output preview

When writing is enabled, reopen the exact file G2 will read and validate its source, document ID, and segment coverage. Print only document ID, counts, checkpoint names, and destination; never print the segment text or prediction labels.

In [19]:
if WRITE_PREDICTIONS:
    from reflection_assessment_feedback.experiment_config import experiment_config_sha256

    if set(OUTPUT_PATHS) != set(DEVELOPMENT_DOCUMENT_IDS):
        raise RuntimeError("Configure protected output paths for every development document.")
    saved_analyses = {}
    for document_id, output_path in OUTPUT_PATHS.items():
        saved_package = json.loads(output_path.read_text(encoding="utf-8"))
        if saved_package["document_id"] != document_id:
            raise ValueError("Saved prediction document ID does not match its artifact.")
        if saved_package.get("experiment_config_sha256") != experiment_config_sha256():
            raise ValueError("Saved prediction config fingerprint does not match this run.")
        saved_analysis = IntermediateAnalysis.model_validate(saved_package["analysis"])
        if saved_analysis.source != "predicted":
            raise ValueError("Saved analysis must have source='predicted'.")
        if len(saved_analysis.segments) != len(DEVELOPMENT_DOCUMENTS[document_id].segments):
            raise ValueError("Saved prediction does not cover every source segment.")
        make_generation_request(
            condition="G2",
            document=DEVELOPMENT_DOCUMENTS[document_id],
            rubric=EXPERT_RUBRIC,
            analysis=saved_analysis,
        )
        saved_analyses[document_id] = saved_analysis
    print(f"All {len(saved_analyses)} prediction artifacts passed document, coverage, and config checks.")
else:
    print("No prediction artifacts to preview; inference or writing is disabled.")

All 6 prediction artifacts passed document, coverage, and config checks.
